In [81]:
import satpy
from satpy.scene import Scene
from satpy import find_files_and_readers
from datetime import datetime
import os
import pandas as pd
import rasterio as rio
import pyproj
import numpy as np
from pyresample import create_area_def
import pyspectral

In [37]:
dataloc = "/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024/"

In [38]:
files = [f for f in os.listdir(dataloc) if '.hdf' in f]

def timestamp_from_modis_fname(fname):
    shortname, juliandate, hourmin, collection, juliandatetime, filetype = fname.split('.')
    hour = int(hourmin[0:2])
    minute = int(hourmin[2:4])
    julianday = int(juliandate[5:])
    year = int(juliandate[1:5])
    return pd.to_datetime(str(year) + '-01-01 00:00') + pd.to_timedelta(julianday + hour/24 + minute/(24*60) - 1, unit='D').round('1min')

def satellite_from_modis_fname(fname):
    if "MYD" in fname:
        return "Aqua"        
    elif "MOD" in fname:
        return "Terra"

def resolution_from_modis_fname(fname):
    if "21KM" in fname:
        return "1000m"
    elif "2HKM" in fname:
        return "500m"
    elif "2QKM" in fname:
        return "250m"
    else:
        return "GEO"
        
df_granules = pd.DataFrame(
    {'datetime': [timestamp_from_modis_fname(f) for f in files],
     'satellite': [satellite_from_modis_fname(f) for f in files],
     'resolution': [resolution_from_modis_fname(f) for f in files],
     'filename': files}
)
df_granules = df_granules.sort_values("datetime").reset_index(drop=True)
df_granules

,datetime,satellite,resolution,filename
0,2024-06-28 03:30:00,Aqua,GEO,MYD03.A2024180.0330.061.2024180193848.hdf
1,2024-06-28 03:30:00,Aqua,250m,MYD02QKM.A2024180.0330.061.2024180200732.hdf
2,2024-06-28 03:30:00,Aqua,500m,MYD02HKM.A2024180.0330.061.2024180200732.hdf
3,2024-06-28 05:05:00,Aqua,250m,MYD02QKM.A2024180.0505.061.2024180230327.hdf
4,2024-06-28 05:05:00,Aqua,500m,MYD02HKM.A2024180.0505.061.2024180230327.hdf
...,...,...,...,...
91,2024-06-29 20:05:00,Terra,GEO,MOD03.A2024181.2005.061.2024182030544.hdf
92,2024-06-29 20:05:00,Terra,250m,MOD02QKM.A2024181.2005.061.2024182073834.hdf
93,2024-06-29 21:45:00,Terra,250m,MOD02QKM.A2024181.2145.061.2024182073725.hdf
94,2024-06-29 21:45:00,Terra,GEO,MOD03.A2024181.2145.061.2024182030801.hdf


In [39]:
reference_img = rio.open("../data/MODIS-reference-2024-06-29.tiff")
left, bottom, right, top = reference_img.bounds

proj_LL = 'epsg:4326' # WGS 84 Ellipsoid
proj_XY = 'epsg:3413' # NSIDC Polar Stereographic
transform_to_ll = pyproj.Transformer.from_crs(proj_XY, proj_LL, always_xy=True)

lower_left_lon, lower_left_lat = transform_to_ll.transform(left, bottom)
upper_right_lon, upper_right_lat = transform_to_ll.transform(right, top)

dx = np.round(right - left)
dy = np.round(top - bottom)
dx = np.round(dx, -3)
dy = np.round(dy, -3)
x0 = np.round(left)
y0 = np.round(bottom)
integer_extents = (x0, y0, x0 + dx, y0 + dy)

area_id = 'nsidc_polar_north'
description = 'NSIDC Sea Ice Polar Stereographic North'
projection = 'EPSG:3413' 
resolution = 250
area = create_area_def(area_id=area_id, description=description, projection=projection, 
                resolution=resolution, area_extent=integer_extents)    

In [40]:
area.area_extent

(693980.0, -2067418.0, 1029980.0, -1375418.0)

In [41]:
myfiles

{'modis_l1b': ['/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024/MYD02QKM.A2024180.0505.061.2024180230327.hdf',
  '/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024/MYD02HKM.A2024180.0505.061.2024180230327.hdf',
  '/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024/MYD03.A2024180.0505.061.2024180205115.hdf']}

In [47]:
day_files

{'modis_l1b': ['/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024//MYD03.A2024180.0330.061.2024180193848.hdf',
  '/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024//MYD02QKM.A2024180.0330.061.2024180200732.hdf',
  '/Users/dwatkin2/Documents/research/data/modis_granules/greenland_sea_2024//MYD02HKM.A2024180.0330.061.2024180200732.hdf']}

In [98]:
for date, group in df_granules.groupby('datetime'):
    day_files = {'modis_l1b': [dataloc + f for f in group.filename.values]}
    scn = Scene(filenames=day_files)
    scn.load(['1', '3', '4'])
    scn.load(["true_color_crefl"])
    unified_scn = scn.resample(area, resampler="nearest")
    unified_scn.save_dataset("true_color_crefl", "../data/truecolor/modis.truecolor.{d}.{s}.png".format(
        d=date.strftime("%Y%m%d.%H%M"), s=group.satellite.values[0].lower())
    )
    del scn, unified_scn

The following datasets were not created and may require resampling to be generated: DataID(name='true_color_crefl')
/opt/miniconda3/envs/ift-eddies/lib/python3.14/site-packages/satpy/modifiers/_crefl_utils.py:534: RuntimeWarning: invalid value encountered in log
  return np.exp(-np.exp(self._ah2o + self._bh2o * np.log(self._airmass * UH2O_MODIS)))
The following datasets were not created and may require resampling to be generated: DataID(name='true_color_crefl')
The following datasets were not created and may require resampling to be generated: DataID(name='true_color_crefl')
/opt/miniconda3/envs/ift-eddies/lib/python3.14/site-packages/satpy/modifiers/_crefl_utils.py:534: RuntimeWarning: invalid value encountered in log
  return np.exp(-np.exp(self._ah2o + self._bh2o * np.log(self._airmass * UH2O_MODIS)))
The following datasets were not created and may require resampling to be generated: DataID(name='true_color_crefl')
The following datasets were not created and may require resampling t